In [9]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
print("Installing bcftools...")
!sudo apt-get update -q
!sudo apt-get install bcftools -y -q

Installing bcftools...
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.8 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,887 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,673 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [62.6 kB]
Get:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRe

In [ ]:
import os
import subprocess

In [ ]:
INPUT_DIR = "/content/drive/MyDrive/FYP_DATA/DATA/raw_data/SG10K/"
ALIAS_FILE = "/content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/indian_alias.txt"
OUTPUT_DIR = "/content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered/"

In [ ]:
# # ONE TIME FIX, DONOT DO THIS AGAIN
# print("🔧 Fixing line endings in alias file...")
# subprocess.run(f"sed -i 's/\r$//' '{ALIAS_FILE}'", shell=True, check=True)
# print("✅ Alias file converted to Unix format.")

🔧 Fixing line endings in alias file...
✅ Alias file converted to Unix format.


In [ ]:
# --- CONFIGURATION ---
chr_num = "1"
# Adjust filename pattern to match yours (e.g., "chr1.vcf.gz" or "sg10k_chr1.vcf.gz")
input_vcf = os.path.join(INPUT_DIR, f"chr{chr_num}.vcf.gz")
output_vcf = os.path.join(OUTPUT_DIR, f"chr{chr_num}_sas_filtered.vcf.gz")

print(f"Processing Chromosome {chr_num}...")
print(f"Input: {input_vcf}")
print(f"Output: {output_vcf}")

# --- VERIFICATION: BEFORE RUNNING ---
print("📊 Pre-processing verification:")
print(f"Total samples in original: {os.popen(f'bcftools query -l {input_vcf} | wc -l').read().strip()}")
print(f"Indian samples requested: {os.popen(f'wc -l < {ALIAS_FILE}').read().strip()}")
print()

# --- THE COMMAND ---
# 1. -S: Keep only Indian samples
# 2. --force-samples: Ignore if ID missing (safety)
# 3. +fill-tags: Recalculate AF, AC, AN for the new subset
# 4. --threads 2: Use Colab's 2 cores

cmd = f"""
bcftools view -S "{ALIAS_FILE}" --force-samples -Ou "{input_vcf}" | \
bcftools plugin fill-tags -Oz -o "{output_vcf}" -- -t AF,AC,AN
"""

# Run it
print("⏳ Running bcftools...")
exit_code = os.system(cmd)
print()

if exit_code == 0:
    print(f"✅ Success! Created: {output_vcf}")

    # Index the new file immediately
    print("Indexing...")
    os.system(f"bcftools index -t '{output_vcf}'")

    # Check size
    size_mb = os.path.getsize(output_vcf) / (1024 * 1024)
    print(f"File Size: {size_mb:.2f} MB")

    # --- VERIFICATION: AFTER RUNNING ---
    print()
    print("📊 Post-processing verification:")
    n_samples = os.popen(f"bcftools query -l '{output_vcf}' | wc -l").read().strip()
    print(f"Final sample count: {n_samples}")

else:
    print("❌ Error processing file. Check paths and logs.")

Processing Chromosome 1...
Input: /content/drive/MyDrive/FYP_DATA/DATA/raw_data/SG10K/chr1.vcf.gz
Output: /content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered/chr1_sas_filtered.vcf.gz
📊 Pre-processing verification:
Total samples in original: 4810
Indian samples requested: 1124

⏳ Running bcftools...

✅ Success! Created: /content/drive/MyDrive/FYP_DATA/DATA/processed_data/SG10K/sas_filtered/chr1_sas_filtered.vcf.gz
Indexing...
File Size: 522.98 MB

📊 Post-processing verification:
Final sample count: 1125


# Quality Check for the First Chromosome, so the same can be applied to other chr files as well.

In [ ]:
# # --- CONFIGURATION ---
# filtered_vcf = os.path.join(OUTPUT_DIR, f"chr1_sas_filtered.vcf.gz")

# print("=" * 80)
# print("VERIFICATION OF FILTERED SG10K CHR1 (Indian samples only)")
# print("=" * 80)

# # ==========================================
# # 1. BASIC FILE CHECKS
# # ==========================================
# print("\n📁 1. FILE INFORMATION:")
# print("-" * 80)
# if os.path.exists(filtered_vcf):
#     size_mb = os.path.getsize(filtered_vcf) / (1024 * 1024)
#     print(f"✅ File exists: {filtered_vcf}")
#     print(f"   File size: {size_mb:.2f} MB")

#     # Check index
#     if os.path.exists(filtered_vcf + ".tbi"):
#         index_size = os.path.getsize(filtered_vcf + ".tbi") / (1024 * 1024)
#         print(f"✅ Index exists: {index_size:.2f} MB")
#     else:
#         print("❌ Index missing!")
# else:
#     print(f"❌ File not found: {filtered_vcf}")

# # ==========================================
# # 2. SAMPLE COUNT VERIFICATION
# # ==========================================
# print("\n📊 2. SAMPLE COUNT VERIFICATION:")
# print("-" * 80)

# # Count samples in filtered VCF
# n_filtered = int(os.popen(f"bcftools query -l '{filtered_vcf}' | wc -l").read().strip()) + 1
# # Count samples in alias file
# n_expected = int(os.popen(f"wc -l < '{ALIAS_FILE}'").read().strip())

# print(f"Expected (from alias file): {n_expected}")
# print(f"Actual (in filtered VCF):   {n_filtered}")

# if n_filtered == n_expected:
#     print(f"✅ MATCH! All {n_filtered} Indian samples present")
# else:
#     print(f"⚠️  MISMATCH! Expected {n_expected} but got {n_filtered}")

# # Show first 10 sample IDs
# print("\nFirst 10 sample IDs in filtered VCF:")
# !bcftools query -l "{filtered_vcf}" | head -10

# # ==========================================
# # 3. VARIANT COUNT COMPARISON
# # ==========================================
# print("\n🧬 3. VARIANT COUNT:")
# print("-" * 80)

# original_vcf = os.path.join(INPUT_DIR, "chr1.vcf.gz")
# n_variants_orig = int(os.popen(f"bcftools view -H '{original_vcf}' | wc -l").read().strip())
# n_variants_filt = int(os.popen(f"bcftools view -H '{filtered_vcf}' | wc -l").read().strip())

# print(f"Original VCF variants: {n_variants_orig:,}")
# print(f"Filtered VCF variants: {n_variants_filt:,}")
# print(f"Difference: {n_variants_orig - n_variants_filt:,} variants removed")
# print(f"Retention rate: {(n_variants_filt/n_variants_orig)*100:.2f}%")

# # ==========================================
# # 4. INFO FIELD VERIFICATION (AF, AC, AN)
# # ==========================================
# print("\n📈 4. INFO FIELD VERIFICATION (AF, AC, AN):")
# print("-" * 80)

# # Check if AF, AC, AN are present and calculated correctly
# print("Checking first 10 variants with AF, AC, AN values:\n")
# !bcftools query -f '%CHROM:%POS\t%REF\t%ALT\tAC=%AC\tAN=%AN\tAF=%AF\n' "{filtered_vcf}" | head -10

# # ==========================================
# # 5. ALLELE FREQUENCY SANITY CHECK
# # ==========================================
# print("\n🔢 5. ALLELE FREQUENCY SANITY CHECKS:")
# print("-" * 80)

# # Expected AN = 2 * number of samples (for diploid)
# expected_an = 2 * n_filtered
# print(f"Expected AN (2 × {n_filtered} samples): {expected_an}")

# # Check actual AN values
# print("\nChecking AN values in first 100 variants:")
# an_check = os.popen(f"bcftools query -f '%AN\\n' '{filtered_vcf}' | head -100 | sort | uniq -c").read()
# print(an_check)

# # Check AF range (should be between 0 and 1)
# print("\nChecking AF range (should be 0-1):")
# !bcftools query -f '%AF\n' "{filtered_vcf}" | head -1000 | awk '{{if($1<0 || $1>1) print "⚠️  Invalid AF:", $1; count++}} END {{if(count==0) print "✅ All AF values in valid range (0-1)"}}'

# # ==========================================
# # 6. COMPARE AF: ORIGINAL vs FILTERED
# # ==========================================
# print("\n🔄 6. ALLELE FREQUENCY COMPARISON (Original vs Filtered):")
# print("-" * 80)
# print("Comparing AF values for first 10 variants:\n")

# # Get AF from original
# !bcftools query -f '%CHROM:%POS\t%AF\n' "{original_vcf}" | head -10 > /tmp/original_af.txt

# # Get AF from filtered
# !bcftools query -f '%CHROM:%POS\t%AF\n' "{filtered_vcf}" | head -10 > /tmp/filtered_af.txt

# # Display comparison
# print("Position\t\tOriginal_AF\tFiltered_AF")
# !paste /tmp/original_af.txt /tmp/filtered_af.txt | awk '{{print $1"\t"$2"\t\t"$4}}'

# print("\n💡 Note: AF values SHOULD be different (Indian-only vs all populations)")

# # ==========================================
# # 7. GENOTYPE CHECK
# # ==========================================
# print("\n🧬 7. GENOTYPE DATA CHECK:")
# print("-" * 80)
# print("First variant with genotypes for first 5 samples:\n")
# !bcftools query -f '%CHROM:%POS\t%REF\t%ALT[\t%GT]\n' "{filtered_vcf}" | head -1 | cut -f1-8

# # ==========================================
# # 8. FINAL SUMMARY
# # ==========================================
# print("\n" + "=" * 80)
# print("✅ VERIFICATION SUMMARY")
# print("=" * 80)
# print(f"Sample count: {n_filtered}/{n_expected} {'✅' if n_filtered == n_expected else '❌'}")
# print(f"Variant count: {n_variants_filt:,}")
# print(f"File size: {size_mb:.2f} MB")
# print(f"Index: {'✅' if os.path.exists(filtered_vcf + '.tbi') else '❌'}")
# print("\nIf all checks pass, you can proceed with batch processing chr2-22!")
# print("=" * 80)